In [ ]:
#Import the needed libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
#For the CNN model:
import tensorflow as tf
import pickle
from tensorflow .keras import Sequential
from tensorflow.keras.layers import     Conv2D, MaxPooling2D, Flatten, Dense, Dropout ,BatchNormalization,Activation ,Input  ,GlobalAveragePooling2D

In [ ]:
#import the required csv file
mycsvfile = pd.read_csv('messidor_data.csv')
#Read required columns
Copymycsvfile=mycsvfile[['id_code','diagnosis']]
#Convert the multiclass diagnosis to binary class
# Images with diagnosis grade 1, 2, 3 or 4 are mapped to 1 = DR present
#Images with diagnosis grade 0 are mapped to 0 = No DR
Copymycsvfile['diagnosis'] = (
    Copymycsvfile['diagnosis'] > 0
).astype(int)
print(Copymycsvfile['diagnosis'].value_counts())

In [ ]:
#Create array to load images into
impgsarray = []
#create array to load diagnosis for each image
labelsarray = []
import os
string = r'C:\Users\Aphiwe Madondo\anaconda_projects\FinalMachineLearningProject\preprocess'
#Function to load the image into the array
def load_images_from_folder(folder, dataframe):
    total = len(dataframe)
    #Loop through images in folder
    for i, (_, row) in enumerate(dataframe.iterrows()):
        img_path = folder + '\\' + row['id_code']   # builds full path from CSV
        img = cv2.imread(img_path, cv2.IMREAD_COLOR) # fix: IMREAD_COLOR not IMREAD_COLOR_RGB
         #if is a valid image
        if img is not None:
            #convert the image into RGB format
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            #Convert image to size 224 x 224
            img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_LINEAR)
            #Normalize the values in RGB ,this converts the range from [0,255] to [ 0,1]
            img = img / 255.0
            #Converts image from integer type to 32-bit floating point.
            img = img.astype(np.float32)
            #add image to array
            impgsarray.append(img)
            #add diagnosis to array
            labelsarray.append(row['diagnosis'])

        if i % 100 == 0:
            print(f"Progress: {i}/{total} images loaded")

load_images_from_folder(string, Copymycsvfile)
print(f"Done! Loaded {len(impgsarray)} images")

In [ ]:
#Apply CLAHE to each image
def apply_clahe(img_float):
    # convert back to uint8 for CLAHE
    img_uint8 = (img_float * 255).astype(np.uint8)
    #create CLAHE object :
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    #Split images into 3 different RGB channels
    channels = list(cv2.split(img_uint8))
    #apply CLAHE to first channel
    channels[0] = clahe.apply(channels[0])
    #apply CLAHE to second channel
    channels[1] = clahe.apply(channels[1])
    #apply CLAHE to third channel
    channels[2] = clahe.apply(channels[2])
    #combine processed channels
    img_clahe = cv2.merge(channels)
    # back to float32
    return img_clahe.astype(np.float32) / 255.0
#Function to brighten dark images
def brighten_dark_image(img):
    brightness = np.mean(img)
    if brightness < 0.4:
        img = np.clip(img * 1.5, 0, 1)
    return img

# Apply to all images
X_processed = np.array([brighten_dark_image(apply_clahe(img)) for img in impgsarray])
print(f"Done! Shape: {X_processed.shape}, dtype: {X_processed.dtype}")



In [ ]:
#Split the dataset into 80 % training and20%  testing sets
from  sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(np.array(X_processed),np.array(labelsarray),random_state=0,test_size=0.2,stratify=np.array(labelsarray))


In [ ]:
#Custom CNN model with 11 layers
def build_cnn_model():
    model = Sequential()
    #give model input
    model.add(Input(shape=(224, 224, 3)))
    #layer 1
    model.add(Conv2D(32, (3,3), padding='same', activation='relu'))
    #layer 2
    model.add(MaxPooling2D(pool_size=(2,2)))
    #layer 3
    model.add(Conv2D(64, (3,3), padding='same', activation='relu'))
    #layer 4
    model.add(MaxPooling2D(pool_size=(2,2)))
    #layer 5
    model.add(Conv2D(128, (3,3), padding='same', activation='relu'))
    #layer 6
    model.add(MaxPooling2D(pool_size=(2,2)))
    #layer 7
    model.add(Conv2D(256, (3,3), padding='same', activation='relu'))
    #layer 8
    model.add(MaxPooling2D(pool_size=(2,2)))
    #layer 9
    model.add(GlobalAveragePooling2D())
    #layer 10
    model.add(Dense(256, activation='relu'))
    model.add(Dropout(0.5))
    #layer 11
    model.add(Dense(1, activation='sigmoid'))

    return model

In [ ]:
from tensorflow.keras.losses import BinaryFocalCrossentropy
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau
# ===============Step 1 — Augmentation=====================#
#Create an image generator to apply random augmentation
Mytrain_DataGenerator = ImageDataGenerator(
    #Randomly rotate images by -+15 degrees
    rotation_range=15,
    #Randomly shift image horizontally up to 10% of image width
    width_shift_range=0.1,
    #Randomy shift images vertically up to 10% of image height
    height_shift_range=0.1,
    #Randomly flip images horizonatlly
    horizontal_flip=True,
    #zoom up to 10%
    zoom_range=0.1,
    #Fill empty pixels created by transformations
    fill_mode='nearest'
)
#generate batches of augmented trianing images
train_gen = Mytrain_DataGenerator.flow(X_train, Y_train, batch_size=32, shuffle=True)

#================== Step 2 — Build and train==================
#Create CNN model
model = build_cnn_model()
#Compile model for training
#Binary focal cross entropy places more emphasis on difficult examples and can improve performance on imbalanced datasets.

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=BinaryFocalCrossentropy(gamma=2.0),
    metrics=['accuracy']
)
#Train the model
#evaluate performance on the validation set after each epoch
modelperformance = model.fit(
    train_gen,
    validation_data=(X_test, Y_test),
    epochs=50,
    callbacks=[
        #Stop training if validation loss does not improve for 15 epochs and restore the best performing model weights.
        EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
          # Reduce the learning rate when validation loss plateaus
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
    ]
)

In [ ]:

#Build the model once, then remove the final classification layer to us CNN as a feature extractor
_ = model(X_train[:1])

feature_extractor = tf.keras.Sequential(model.layers[:-1])
#extract thelearned feature representations from the training and test images.
X_train_features = feature_extractor.predict(X_train)
X_test_features = feature_extractor.predict(X_test)
print("Feature shape:", X_train_features.shape)

In [ ]:

# — SVM classifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
#Train a  Support vector machine  using the features that are extracted by the  CNN
#Class weighting is used to hanlde class imbalance
svm = SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced')
#fit the model on extracted features
svm.fit(X_train_features, Y_train)
y_pred = svm.predict(X_test_features)

# Results
print("=" * 45)
print("=SVM=")
print(f"  Test Accuracy  : {accuracy_score(Y_test, y_pred)*100:.2f}%")
print(f"  Precision      : {precision_score(Y_test, y_pred)*100:.2f}%")
print(f"  Recall         : {recall_score(Y_test, y_pred)*100:.2f}%")
print(f"  F1-Score       : {f1_score(Y_test, y_pred)*100:.2f}%")
print("=" * 45)
#
print(classification_report(Y_test, y_pred,
      target_names=['No DR', 'DR'], zero_division=0))
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
#output confusion matrix
cm = confusion_matrix(Y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No DR', 'DR'])

disp.plot(cmap='Blues')
plt.title('CNN Feature Extraction+Support Vector Machine Classifier — Confusion Matrix')
plt.show()

In [ ]:
#Plot training and validation accuracy and loss to evaluate model performance
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Accuracy
axes[0].plot(modelperformance.history['accuracy'], label='Train Accuracy', color='blue')
axes[0].plot(modelperformance.history['val_accuracy'], label='Val Accuracy', color='orange')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(modelperformance.history['loss'], label='Train Loss', color='blue')
axes[1].plot(modelperformance.history['val_loss'], label='Val Loss', color='orange')
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.suptitle('Training vs Validation Curves', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
from tensorflow.keras.models import Model
import numpy as np

# Pick 4 random samples from test set
Randomimages= np.random.choice(len(X_test), 4, replace=False)
Random_sample_images = X_test[Randomimages]
sample_true = Y_test[Randomimages]

# Extract features using the same method used during SVM training
sample_features = X_test_features[Randomimages]
sample_pred = svm.predict(sample_features)

# Plot 2 rows x 2 columns
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for i, ax in enumerate(axes):
    ax.imshow(Random_sample_images[i])

    true_label = 'DR' if sample_true[i] == 1 else 'No DR'
    pred_label = 'DR' if sample_pred[i] == 1 else 'No DR'
    correct = true_label == pred_label

    ax.set_title(
        f"True: {true_label}\nPred: {pred_label}",
        color='green' if correct else 'red',
        fontsize=12
    )
    ax.axis('off')

plt.suptitle('Custom CNN + SVM — Sample Retinal Image Predictions\n(Green=Correct, Red=Wrong)', fontsize=13)
plt.tight_layout()
plt.show()